# Indic Document Intelligence with Kimi K3 on Amazon Bedrock

This notebook shows how to use **Kimi K3** on **Amazon Bedrock** to read scanned Hindi documents and turn them into structured data, using the **India cross-Region inference profile** (`in.moonshotai.kimi-k3`) so that inference stays within India (Mumbai `ap-south-1` and Hyderabad `ap-south-2`).

You will:

1. Call Kimi K3 directly with the OpenAI Python SDK on the `bedrock-runtime` endpoint's [Chat Completions API](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-chat-completions.html), as the [Kimi K3 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-moonshot-ai-kimi-k3.html) recommends, to see what the model gives you on its own for a scanned Hindi electricity bill.
2. Define a **schema** for the bill and use the sample's small extraction pipeline to get back a full transcription, structured fields as JSON, and an English summary.
3. Check the extracted fields against hand-labelled ground truth.
4. Ask follow-up questions about the same document in **Hinglish** (code-mixed Hindi-English).
5. Repeat the flow on a hand-filled Hindi KYC form with a second schema, and on any document of your own with no schema at all.

> **Note:** All documents in `sample_data/` are **synthetic**. Every organisation, person, address and number is fictional, and identity numbers are masked. Do not use real personal documents with this sample unless you have the right to process them.

## Prerequisites

- An AWS account with access to Kimi K3 in Amazon Bedrock in `ap-south-1`. See [Access Amazon Bedrock foundation models](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html).
- IAM permission for `bedrock:InvokeModel` on the `in.moonshotai.kimi-k3` inference profile and the underlying foundation model in both India Regions, plus `bedrock:CallWithBearerToken`. The sample turns your AWS credentials into a short-term Amazon Bedrock API key. See the README for an example policy.
- Python 3.12 or later, and AWS credentials configured for this environment.

## Setup

You'll need to install the required libraries and **select your Python kernel environment** to run this notebook.

**For local IDEs**, we recommend using [uv](https://docs.astral.sh/uv/) to manage your Python environments. After installing it, you can open a terminal in this repository's root folder and run:
1. `uv venv --allow-existing` to create the virtual environment (usually in a new folder `.venv` in the project itself), then
2. `uv sync` to install the libraries.

When the virtual environment is set up, select it as the kernel for this notebook (In VS Code or Kiro, choose `Python Environments > .venv` when prompted).

You can also use plain `venv` if you prefer, and `pip install -e .` to install the dependencies (which are specified in this repository's top-level [pyproject.toml](../../../pyproject.toml)).

**For SageMaker JupyterLab Notebooks**, you can use the base `conda_python3` kernel and install the dependencies through pip, by un-commenting and running the code cell below:

In [ ]:
# (For SageMaker JupyterLab)
# %pip install -e ../../..

Imports the sample code and loads the settings (model ID, Region, reasoning effort) used by every call in this notebook.

In [2]:
import json
from pathlib import Path

from indic_doc_intel import IndicDocumentIntelligence, Settings, load_image
from indic_doc_intel.evaluation import compare_fields, load_expected

# Folder with the synthetic sample documents and their ground-truth labels.
SAMPLE_DATA = Path.cwd() / "sample_data"

# Defaults: model in.moonshotai.kimi-k3, Region ap-south-1, model-default reasoning effort (max).
# Override with KIMI_MODEL_ID, BEDROCK_REGION and KIMI_REASONING_EFFORT, or pass
# Settings(reasoning_effort="low") to trade a little quality for fewer output tokens.
settings = Settings.from_env()
print(f"Model: {settings.model_id}\nRegion: {settings.region}")

Model: in.moonshotai.kimi-k3
Region: ap-south-1


Defines two small display helpers used in later cells: `show_json` prints results, and `show_comparison` prints a field-by-field ground-truth check.

In [3]:
def show_json(data):
    """Pretty-print a dict, keeping Hindi text readable (ensure_ascii=False)."""
    print(json.dumps(data, ensure_ascii=False, indent=2))


def show_comparison(rows):
    """Print one line per field: OK if the extracted value matches ground truth, DIFF otherwise."""
    width = max(len(row["field"]) for row in rows)
    for row in rows:
        mark = "OK  " if row["match"] else "DIFF"
        print(f"{mark}  {row['field']:<{width}}  expected={row['expected']!r:<30} extracted={row['extracted']!r}")
    # Summary line, e.g. "8/8 fields match ground truth"
    matched = sum(row["match"] for row in rows)
    print(f"\n{matched}/{len(rows)} fields match ground truth")

## Step 1: Look at the sample document

A synthetic electricity bill (बिजली बिल) from a fictional distribution company in Jaipur. Note that the dates and the units consumed use **Devanagari digits** (e.g. `०५/१०/२०२६`, `२८८`), while other numbers use Western digits, as is common on real Indian documents.

<img src="sample_data/electricity_bill_hi.png" width="520" alt="Synthetic Hindi electricity bill">

## Step 2: Call Kimi K3 directly

Before adding any structure, here is a plain Kimi K3 call. Amazon Bedrock exposes an OpenAI-compatible Chat Completions endpoint on `bedrock-runtime`, so the standard OpenAI Python SDK works once you point `base_url` at it. `provide_token()` turns your AWS credentials into a short-term Amazon Bedrock API key locally, so no long-term key is needed.

The only helper used here is `load_image()`, which downsizes the scan if needed and base64-encodes it as an image content part.

In [4]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

bill_image = SAMPLE_DATA / "electricity_bill_hi.png"

# The standard OpenAI SDK, pointed at the OpenAI-compatible endpoint of bedrock-runtime.
client = OpenAI(
    base_url=f"https://bedrock-runtime.{settings.region}.amazonaws.com/openai/v1",
    api_key=provide_token(region=settings.region),
)

response = client.chat.completions.create(
    model=settings.model_id,
    messages=[
        {
            "role": "user",
            # The model card recommends placing images before text.
            "content": [
                load_image(bill_image),
                {"type": "text", "text": "Read this document and extract its key fields as JSON."},
            ],
        }
    ],
    # Counts reasoning tokens as well as the answer, so leave generous headroom.
    max_completion_tokens=settings.max_tokens,
)
print(response.choices[0].message.content)

```json
{
  "document_type": "electricity_bill",
  "is_sample_document": true,
  "utility": {
    "name_hi": "काल्पनिक विद्युत वितरण निगम लिमिटेड",
    "name_en": "Kalpanik Vidyut Vitran Nigam Ltd.",
    "note": "काल्पनिक संस्था — केवल प्रदर्शन हेतु",
    "customer_care": "1800-000-0000",
    "website": "www.example.com"
  },
  "consumer": {
    "name": "श्रीमती सुनीता देवी शर्मा",
    "consumer_number": "4102 7789 3561",
    "address": "मकान नं. 27, गली नं. 4, शास्त्री नगर, जयपुर, राजस्थान – 302016",
    "connection_category": "घरेलू (एकल फेज़)",
    "sanctioned_load_kw": 3
  },
  "billing": {
    "bill_date": "2026-10-05",
    "bill_date_original": "०५/१०/२०२६",
    "period_from": "2026-09-01",
    "period_to": "2026-09-30",
    "period_original": "०१/०९/२०२६ से ३०/०९/२०२६",
    "due_date": "2026-10-20",
    "due_date_original": "२०/१०/२०२६"
  },
  "meter": {
    "meter_number": "JPR-55120934",
    "previous_reading": 12450,
    "current_reading": 12738,
    "total_consumption_units"

Kimi K3 reads the Hindi bill well on its own. For an application that consumes the output, though, a bare call leaves several gaps:

- **Field names are up to the model**, so they can change between runs and documents, and downstream code can't rely on them.
- **Formats aren't normalised.** Dates, Devanagari digits and amounts come back however the model chooses.
- **The JSON may be wrapped** in markdown fences or prose, and nothing retries if it doesn't parse.
- **No transcription, summary or uncertainty flags** unless you ask for them every time.
- **Nothing to compare against ground truth**, so prompt or model changes are hard to evaluate.

The rest of this notebook adds a small prompt pipeline over the same call ([`indic_doc_intel/pipeline.py`](indic_doc_intel/pipeline.py)). It adds a system prompt with extraction rules, builds the prompt from a schema you define, parses the JSON (asking the model to repair it once if needed), fills in missing fields as `null`, keeps the conversation for follow-up questions, adds up token usage and refreshes the API key before each request.

## Step 3: Define a schema for the document

A schema is how you tell the pipeline what to extract. It is a plain dict with a `description` of the document type and a `fields` dict that maps each output key to a short instruction, which is passed to the model verbatim. Writing a schema is the main thing you do to support a new document type, so it lives here in the notebook rather than in the library.

In [5]:
ELECTRICITY_BILL_SCHEMA = {
    "description": "An Indian electricity (bijli) bill issued by a state power distribution company.",
    "fields": {
        "distribution_company": "Name of the electricity board or distribution company, in English.",
        "consumer_name": "Account holder name exactly as printed, in the original script.",
        "consumer_name_english": "Account holder name transliterated into English (Latin script).",
        "consumer_number": "Consumer / account number, digits only.",
        "address": "Supply address translated into English.",
        "billing_period": "Billing period as 'YYYY-MM-DD to YYYY-MM-DD'.",
        "bill_date": "Bill issue date as YYYY-MM-DD.",
        "due_date": "Payment due date as YYYY-MM-DD.",
        "units_consumed_kwh": "Units consumed in this period, as a number.",
        "amount_due_inr": "Total payable amount on or before the due date, as a number.",
        "amount_after_due_date_inr": "Amount payable after the due date, as a number.",
    },
}

## Step 4: Extract structured fields with the pipeline

`pipeline.extract()` makes the same kind of Chat Completions call as Step 2, but builds the prompt from `ELECTRICITY_BILL_SCHEMA` and adds the system prompt's extraction rules. Kimi K3 returns one JSON object containing the transcription, exactly the requested fields (dates normalised to ISO format, Devanagari digits converted, names transliterated) and an English summary.

Creates the pipeline, sends the scanned Hindi bill to Kimi K3 with the schema, and prints the structured fields it extracted.

In [6]:
# Wraps a Kimi K3 client for the bedrock-runtime endpoint, built from the same settings.
pipeline = IndicDocumentIntelligence(settings)

# One call to Kimi K3: sends the image plus an extraction prompt built from the schema,
# and parses the JSON reply into a DocumentResult.
bill = pipeline.extract(bill_image, schema=ELECTRICITY_BILL_SCHEMA)

# Structured fields: ISO dates, Western digits, names transliterated into English.
show_json(bill.fields)

{
  "distribution_company": "Kalpanik Vidyut Vitran Nigam Ltd.",
  "consumer_name": "श्रीमती सुनीता देवी शर्मा",
  "consumer_name_english": "Smt. Sunita Devi Sharma",
  "consumer_number": "410277893561",
  "address": "House No. 27, Gali No. 4, Shastri Nagar, Jaipur, Rajasthan - 302016",
  "billing_period": "2026-09-01 to 2026-09-30",
  "bill_date": "2026-10-05",
  "due_date": "2026-10-20",
  "units_consumed_kwh": 288,
  "amount_due_inr": 2217.2,
  "amount_after_due_date_inr": 2261.54
}


Shows the rest of the extraction result: the detected languages, the English summary and the start of the Hindi transcription.

In [7]:
# The same reply also contains the detected languages, an English summary
# and a full transcription of the document in its original script.
print("Detected languages:", bill.data["detected_languages"])
print("\nEnglish summary:\n", bill.english_summary)
print("\nTranscription (first 600 characters):\n", bill.data["transcription"][:600])

Detected languages: ['hi', 'en']

English summary:
 This is an electricity bill from Kalpanik Vidyut Vitran Nigam Ltd. for consumer Smt. Sunita Devi Sharma at House No. 27, Gali No. 4, Shastri Nagar, Jaipur, Rajasthan - 302016. It covers the billing period 2026-09-01 to 2026-09-30 with a consumption of 288 units, and the total amount due is INR 2217.20. Payment must be made by the due date 2026-10-20; after that, the payable amount rises to INR 2261.54 including a 2% late surcharge. The bill also warns that the electricity connection may be disconnected if two consecutive bills remain unpaid.

Transcription (first 600 characters):
 काल्पनिक विद्युत वितरण निगम लिमिटेड
Kalpanik Vidyut Vitran Nigam Ltd. (काल्पनिक संस्था — केवल प्रदर्शन हेतु)
ग्राहक सेवा: 1800-000-0000 | www.example.com

बिजली बिल / विद्युत उपभोग देयक

उपभोक्ता का नाम	श्रीमती सुनीता देवी शर्मा
उपभोक्ता संख्या	4102 7789 3561
पता	मकान नं. 27, गली नं. 4, शास्त्री नगर, जयपुर, राजस्थान – 302016
कनेक्शन श्रेणी	घरेलू (एकल फेज़), 

## Step 5: Check against ground truth

`sample_data/expected/` holds hand-labelled values for each sample. Comparing against them is a simple way to catch regressions when you change prompts or models.

Scores the extraction against the hand-labelled ground truth for the bill.

In [8]:
# Compare the extracted fields with the hand-labelled values in
# sample_data/expected/electricity_bill_hi.json (numbers and spacing are normalised).
show_comparison(compare_fields(bill.fields, load_expected(bill_image)))

OK    consumer_name              expected='श्रीमती सुनीता देवी शर्मा'    extracted='श्रीमती सुनीता देवी शर्मा'
OK    consumer_number            expected='410277893561'                 extracted='410277893561'
OK    billing_period             expected='2026-09-01 to 2026-09-30'     extracted='2026-09-01 to 2026-09-30'
OK    bill_date                  expected='2026-10-05'                   extracted='2026-10-05'
OK    due_date                   expected='2026-10-20'                   extracted='2026-10-20'
OK    units_consumed_kwh         expected=288                            extracted=288
OK    amount_due_inr             expected=2217.2                         extracted=2217.2
OK    amount_after_due_date_inr  expected=2261.54                        extracted=2261.54

8/8 fields match ground truth


## Step 6: Ask follow-up questions in Hinglish

Many users in India type in Hinglish: Hindi words written in Latin script, mixed with English. `pipeline.ask()` continues the same conversation, so Kimi K3 still has the original image in context, and is asked to reply in the user's own language style.

Asks three Hinglish questions about the bill, with one Kimi K3 call per question, and prints each answer.

In [9]:
# Follow-up questions in Hinglish (Hindi written in Latin script, mixed with English).
questions = [
    "Bill kitne ka hai aur last date kab hai? Late hua toh kitna dena padega?",
    "Is mahine kitne units use hue aur meter number kya hai?",
    "Agar main bill pay nahi karun toh kya hoga?",
]

# Each ask() continues the extraction conversation, so the model still sees the bill image,
# and replies in the same style as the question (Hinglish in, Hinglish out).
for question in questions:
    print(f"Q: {question}\nA: {pipeline.ask(bill, question)}\n")

Q: Bill kitne ka hai aur last date kab hai? Late hua toh kitna dena padega?
A: Bill ₹2,217.20 ka hai aur payment ki last date 20/10/2026 hai. Agar late hua toh ₹2,261.54 dena padega (2% late surcharge ke saath).



Q: Is mahine kitne units use hue aur meter number kya hai?
A: Is billing period mein 288 units use hue hain (pichli reading 12450 se current reading 12738 tak) aur meter number JPR-55120934 hai.



Q: Agar main bill pay nahi karun toh kya hoga?
A: Agar aap due date 20/10/2026 ke baad pay karenge toh ₹2,261.54 dena padega (2% late surcharge ke saath). Aur agar lagatar do bills ka payment nahi kiya toh aapka electricity connection kaat diya ja sakta hai.



## Step 7: A hand-filled KYC form

The second sample is a synthetic address-update / KYC form for a fictional cooperative bank, filled in by hand in blue ink, with tick boxes, a masked Aadhaar number and a signature. The same pipeline is used with a second schema for this form.

<img src="sample_data/kyc_form_hi.png" width="520" alt="Synthetic Hindi KYC form">

Defines the schema for the KYC form. Only the document description and field instructions change.

In [10]:
KYC_FORM_SCHEMA = {
    "description": "An Indian address-update / KYC application form filled in Hindi.",
    "fields": {
        "applicant_name": "Applicant name exactly as written, in the original script.",
        "applicant_name_english": "Applicant name transliterated into English (Latin script).",
        "father_or_spouse_name_english": "Father's or spouse's name transliterated into English.",
        "date_of_birth": "Date of birth as YYYY-MM-DD.",
        "gender": "One of 'male', 'female', 'other'.",
        "mobile_number": "10-digit mobile number, digits only.",
        "address": "Full address translated into English.",
        "pin_code": "6-digit PIN code.",
        "id_document_type": "Type of identity document referenced, in English.",
        "id_document_number": "Identity document number exactly as written.",
        "declaration_signed": "true if the declaration is signed, otherwise false.",
    },
}

Extracts the hand-filled KYC form with `KYC_FORM_SCHEMA`, prints the fields, and checks them against ground truth.

In [11]:
kyc_image = SAMPLE_DATA / "kyc_form_hi.png"

# Same pipeline, different schema.
kyc = pipeline.extract(kyc_image, schema=KYC_FORM_SCHEMA)

show_json(kyc.fields)
print()
# Score against sample_data/expected/kyc_form_hi.json
show_comparison(compare_fields(kyc.fields, load_expected(kyc_image)))

{
  "applicant_name": "राजेश कुमार यादव",
  "applicant_name_english": "Rajesh Kumar Yadav",
  "father_or_spouse_name_english": "Ramprasad Yadav",
  "date_of_birth": "1988-03-14",
  "gender": "male",
  "mobile_number": "9876543210",
  "address": "Flat No. 302, Sai Kripa Apartment, Vibhuti Khand, Gomti Nagar, Lucknow, Uttar Pradesh",
  "pin_code": "226010",
  "id_document_type": "Aadhaar (masked)",
  "id_document_number": "XXXX XXXX 4821",
  "declaration_signed": true
}

OK    applicant_name      expected='राजेश कुमार यादव'             extracted='राजेश कुमार यादव'
OK    date_of_birth       expected='1988-03-14'                   extracted='1988-03-14'
OK    gender              expected='male'                         extracted='male'
OK    mobile_number       expected='9876543210'                   extracted='9876543210'
OK    pin_code            expected='226010'                       extracted='226010'
OK    id_document_number  expected='XXXX XXXX 4821'               extracted='XXXX XXX

Asks one Hinglish question about the KYC form and prints the answer.

In [12]:
# A Hinglish follow-up question about the KYC form (continues the KYC conversation).
print(pipeline.ask(kyc, "Is form mein naya address kya likha hai aur kaunsa ID proof diya hai?"))

Form mein naya address likha hai: Flat No. 302, Sai Kripa Apartment, Vibhuti Khand, Gomti Nagar, Lucknow, Uttar Pradesh, PIN code 226010.

ID proof ke roop mein masked Aadhaar diya gaya hai, jiska document number hai XXXX XXXX 4821.


## Step 8: Bring your own document (schema-free mode)

Without a schema, Kimi K3 chooses the most important key-value pairs itself. This is useful for exploring a new document type before you write a schema for it. Point `my_image` at your own (non-sensitive) scanned document to try it.

Runs schema-free extraction. Point `my_image` at your own non-sensitive scan to try a new document type.

In [13]:
my_image = SAMPLE_DATA / "electricity_bill_hi.png"  # replace with your own image path

# No schema: Kimi K3 picks the document type and the key fields itself.
generic = pipeline.extract(my_image)
print("Document type:", generic.data["document_type"])
show_json(generic.fields)

Document type: Electricity bill
{
  "consumer_name": "श्रीमती सुनीता देवी शर्मा",
  "consumer_number": "410277893561",
  "address": "मकान नं. 27, गली नं. 4, शास्त्री नगर, जयपुर, राजस्थान – 302016",
  "connection_category": "घरेलू (एकल फेज़), स्वीकृत भार 3 किलोवाट",
  "bill_date": "2026-10-05",
  "billing_period": "2026-09-01 to 2026-09-30",
  "meter_number": "JPR-55120934",
  "units_consumed": 288,
  "total_amount_due": 2217.2,
  "due_date": "2026-10-20",
  "amount_after_due_date": 2261.54
}


## Token usage

Each `DocumentResult` adds up the token usage reported by the Chat Completions API across extraction and follow-up questions. `completion_tokens` includes `reasoning_tokens`, and `prompt_tokens` includes `cached_prompt_tokens`, which are billed at the lower cache-read rate. Multiply by the current Kimi K3 rates on the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/) to estimate cost.

Prints the token usage of each document so you can estimate cost.

In [14]:
# Each result adds up the tokens used by its extraction call and any follow-up questions.
for name, result in [("electricity_bill", bill), ("kyc_form", kyc), ("generic", generic)]:
    print(f"{name:<17} {result.usage}")

electricity_bill  {'prompt_tokens': 18014, 'completion_tokens': 2385, 'total_tokens': 20399, 'cached_prompt_tokens': 12881, 'reasoning_tokens': 1106}
kyc_form          {'prompt_tokens': 7995, 'completion_tokens': 2538, 'total_tokens': 10533, 'cached_prompt_tokens': 3357, 'reasoning_tokens': 1362}
generic           {'prompt_tokens': 3214, 'completion_tokens': 2961, 'total_tokens': 6175, 'cached_prompt_tokens': 0, 'reasoning_tokens': 1869}


## Clean up

This notebook does not create any AWS resources. You are charged only for the Amazon Bedrock tokens consumed by the calls above.

## Next steps

- Define a schema dict for your own document type, as in Steps 3 and 7, plus a ground-truth file in `sample_data/expected/`.
- Try documents in other Indic languages (Marathi, Tamil, Bengali and others) in schema-free mode.
- For production workloads, add human review for low-confidence fields (see `uncertain_fields` in each result) and store results in a system such as Amazon DynamoDB or Amazon S3.